# Session 4 — What OpenModelica does with a model, and how the solver copes

Guided notebook for Session 4. Canonical lesson: `course/session_04_openmodelica/lesson.md`.

The notebook runs OpenModelica from Python with `energy_training.omc` (a local `omc` if OpenModelica is installed, otherwise the course Docker image). Each run translates the model to C and compiles it first, so it takes a few seconds; section 3 compiles 6 models and runs each with 4 solver settings (about a minute).

1. What the compiler made of M03 and of a wall without heat capacity (states, sorted equations, algebraic loops).
2. Reading the solver statistics of a simulation.
3. Stiffness: M03 with N = 1 ... 50 nodes, implicit and explicit solvers.

Initialization and events are practised in OMEdit with the broken models of exercise G4.1.

In [ ]:
import matplotlib.pyplot as plt

from energy_training.omc.runner import backend_info, simulate_variant
from energy_training.omc.studies import M03_EXAMPLE, SolverCase, storage_solver_study
from energy_training.plotting.style import apply_style
from energy_training.simulation import storage as st
from energy_training.utilities import repo_root

apply_style()
OUT = repo_root() / "output/s04_notebook"  # git-ignored

## 1. From equations to code

The compiler **flattens** the model (one list of variables and equations), then the back end **matches** each equation to the unknown it computes, **sorts** the equations into the order in which they can be solved (block lower triangular form, BLT) and finds the blocks of equations that must be solved together (**algebraic loops**). The debug flag `backenddaeinfo` prints a summary of the result.

**Predict:** how many states and how many algebraic loops will M03 with N = 10 nodes have?

In [ ]:
def summary(report: str) -> str:
    keep = (
        "Number of equations",
        "Number of states",
        "Single equations",
        "Torn equation systems",
        "torn systems",
        "{(",
    )
    lines = [line.strip() for line in report.splitlines() if any(k in line for k in keep)]
    return "\n".join(lines)


m03 = backend_info(M03_EXAMPLE, out_dir=OUT)
print(summary(m03.split("for simulation", 1)[1]))

Ten states (the node temperatures), and every other unknown is computed by one assignment: no algebraic loop. Each node equation can be solved for its derivative, given the states.

`S04_AlgebraicLoops` contains two zones like M01. Zone A loses heat through two wall layers **without heat capacity**; zone B through one layer and then by radiation. The temperature between the layers is not a state, and no single equation gives it: the heat flow rates of the two layers must be equal. **Predict:** what will the compiler report?

In [ ]:
loops = backend_info("EnergyTraining.Examples.S04_AlgebraicLoops", out_dir=OUT)
print(summary(loops.split("for simulation", 1)[1]))
sim_part = loops.split("Algbraic Loops (Simulation)", 1)[1]
sim_loops = sim_part.split("Algbraic Loops (Initialization)")[0]
print(sim_loops.strip(" :#\n"))

Two torn systems, one per zone. **Tearing** picks one unknown, the interface temperature `layer…1.port_b.T` (the *iteration variable*); given a guess of it, the other unknowns of the loop follow one by one (*inner variables*), and one *residual* equation is left to make zero. For zone A the loop is linear: one linear equation for one unknown. For zone B it is nonlinear ($T^4$): it is solved iteratively (Newton), starting from the **start value** of the iteration variable (288.15 K here, the default of `Modelica.Units.SI.Temperature`). Bad start values of iteration variables are a common cause of failed initialisation in larger models.

(`omc` prints *Algbraic* with that spelling.)

## 2. What the solver did: statistics

The runtime flag `-lv=LOG_STATS` makes the simulation executable print timers and counters; `simulate_variant` sets it and `solver_stats` reads it. **Predict:** M03, N = 10, 3 h, DASSL with tolerance 1e-6: how many steps?

In [ ]:
run = simulate_variant(M03_EXAMPLE, {"N": 10}, out_dir=OUT, stop_time=10800.0, interval=60.0)
run.stats

- **steps**: accepted integration steps; DASSL chooses the step size (and order) itself to keep the estimated local error below the tolerance. The 180 output points (`interval` = 60 s) are interpolated and do not force steps.
- **f_calls**: evaluations of the right-hand side, the main cost of a simulation.
- **jacobians**: DASSL is implicit and needs the Jacobian matrix $\partial f/\partial x$ for its Newton iterations; it reuses it over many steps.
- **error test failures**: rejected steps, repeated with a smaller step size.
- **events**: none here (constant inputs).
- **cpu_time**: a few milliseconds; for this model, translating and compiling take far longer than simulating.

## 3. Stiffness: more nodes, faster modes

Each node of M03 empties into the next with the **node time constant** $\tau_n = \rho V/(N\,\dot m)$; the whole tank loses heat with the **standby-loss time constant** $\rho V c_p/UA_{loss}$.

In [ ]:
for n in (1, 10, 50):
    p = st.StorageParams(N=n)
    tau_n = p.m_node / 0.1
    tau_loss = st.loss_time_constant(p)
    print(
        f"N = {n:2d}: tau_n = {tau_n:7.1f} s, loss time constant = {tau_loss / 3600:.0f} h,"
        f" 24 h = {86400 / tau_n:,.0f} tau_n"
    )

The losses barely matter: all eigenvalues of M03 with constant flow equal $-(1/\tau_n + 1/\tau_{loss})$. What matters is that 24 h are hundreds of node time constants. A model is **stiff** when an explicit solver must take steps far smaller than the accuracy requires, because larger steps are unstable. For explicit Euler on a single mode $\dot x = -x/\tau$ the limit is $h < 2\tau$ (Session 1). An **implicit** solver (DASSL, a BDF method) has no such limit for decaying modes and takes steps as large as the accuracy allows.

**Predict:** 24 h of charging, explicit Euler with a step of 300 s. The single-mode limit $2\tau_n$ drops below 300 s at N = 20. What happens at N = 15?

In [ ]:
cases = [
    SolverCase("dassl", 300.0),
    SolverCase("euler", 300.0),
    SolverCase("euler", 60.0),
    SolverCase("rungekutta", 300.0),
]
study = storage_solver_study((1, 2, 5, 10, 15, 20, 50), cases, stop_time=86400.0, out_dir=OUT)
table = study.pivot(index="N", columns="case", values="max_error")
table.insert(0, "2 tau_n [s]", 2 * study.groupby("N")["tau_node"].first())
table.style.format("{:.2g}")

The table gives the largest error of the outlet temperature [K] against the Python M03 solved with a tight tolerance. Explicit Euler with $h$ = 300 s is already wrong by 15 K at N = 10 and by about $10^7$ K at N = 15, although $h < 2\tau_n$ there. In M03 every node is driven by the one upstream: one Euler step maps the temperatures with $(1-r)I + rS$, $r = h/\tau_n$, $S$ the shift to the next node, whose largest row sum $|1-r| + r$ stays at most 1 only for $h \le \tau_n$ (lesson §5.3). Above that, errors are passed down the chain of nodes and grow before they decay. Classical Runge-Kutta (RK4, `rungekutta`) is wrong by about 19 K at N = 15 and blows up at N = 20. With $h$ = 60 s Euler stays bounded up to N = 50, but is inaccurate. DASSL stays near $10^{-3}$ K for every N.

In [ ]:
cost = study.pivot(index="N", columns="case", values="f_calls")
stable = study.pivot(index="N", columns="case", values="max_error") < 1e3
cost.where(stable)

In [ ]:
fig, ax = plt.subplots()
for case in study["case"].unique():
    d = study[(study["case"] == case) & (study["max_error"] < 1e3)]
    ax.loglog(d["f_calls"], d["max_error"], "o-", label=case)
ax.set(
    xlabel="right-hand-side evaluations (24 h)",
    ylabel="max |T_out − reference| [K]",
    title="M03, N = 1 ... 50 (stable runs)",
)
ax.legend(fontsize=8);

DASSL reaches the smallest errors with the fewest evaluations for every N; its cost grows slowly with N (more steps while the temperature front passes through more, sharper layers). The explicit solvers would need steps far below the 60 s used here to be accurate at large N.

**Which solver?** For stiff models (thermal networks with fast and slow parts, discretised tanks and pipes), the implicit default DASSL. Explicit fixed-step solvers are useful when the step is imposed anyway (some co-simulation FMUs, Session 6) — and then the step must respect the stability limit of the fastest mode.

**Tolerance and error.** With tolerance 1e-6 the error of the outlet temperature is about 1e-3 K, not 1e-6 K: the tolerance controls the *local* error of each step, not the error at the end. Exercise **I4.1** measures error and cost against the tolerance, and asks a different question: how many **nodes** are enough? That error comes from the model, not from the solver.